# 02 — Exploratory Data Analysis (EDA)
## STP Natality — Weight Analysis of Newborns

> **Phase 2.2** | Author: STP Natality Team | Last updated: June 2026

This notebook performs a systematic EDA of the US Natality dataset across all 5 year-era files (1986–2018).  
A **sample of 100 000 rows per file (500 000 total)** is used for interactive analysis; the full ~27M-row dataset is processed in `notebooks/03_data_preparation.ipynb`.

### Notebook Structure
1. Setup & Data Loading
2. Dataset Overview
3. Target Variable Analysis (`weight_grams`)
4. Feature Distributions
5. Missing Value Analysis
6. Correlation Analysis
7. Temporal Trend Analysis
8. Data Leakage Check
9. Key Findings Summary

---
## 1. Setup & Data Loading

In [ ]:
# ── Standard imports ───────────────────────────────────────────────────────────
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
from scipy import stats

warnings.filterwarnings('ignore')

# Add project root to path so src.data imports work
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Project imports ────────────────────────────────────────────────────────────
from src.data.ingest import load_all_files, get_data_summary
from src.data.preprocess import run_preprocessing
from src.data.feature_engineering import engineer_features

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_palette('deep')

print(f'pandas {pd.__version__} | numpy {np.__version__}')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set NROWS_PER_FILE = None to load the full dataset (slow but complete)
NROWS_PER_FILE = 100_000   # 500k total sample — fast for interactive EDA
RAW_DATA_DIR   = PROJECT_ROOT / 'newborn_data'

print(f'Loading {NROWS_PER_FILE:,} rows per file from: {RAW_DATA_DIR}')

In [ ]:
# ── Load raw data ──────────────────────────────────────────────────────────────
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

df_raw = load_all_files(
    data_dir=RAW_DATA_DIR,
    nrows_per_file=NROWS_PER_FILE,
)

print(f'\nRaw dataset shape: {df_raw.shape}')
print(f'Memory usage: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# ── Run preprocessing to get clean data for analysis ──────────────────────────
df_clean = run_preprocessing(df_raw)
df_feat  = engineer_features(df_clean)

print(f'After preprocessing: {df_clean.shape}')
print(f'After feature engineering: {df_feat.shape}')

---
## 2. Dataset Overview

In [ ]:
# ── Schema overview ────────────────────────────────────────────────────────────
print('=== RAW DATA SUMMARY ===')
summary = get_data_summary(df_raw)
for k, v in summary.items():
    if k != 'missing_pct_per_col':
        print(f'  {k}: {v}')

In [ ]:
# ── Data types & first rows ────────────────────────────────────────────────────
display(df_raw.dtypes.to_frame('dtype').T)
df_raw.head(3)

In [ ]:
# ── Records per source file ────────────────────────────────────────────────────
file_counts = df_raw['source_file'].value_counts().sort_index()
print('Records per source file (sample):')
display(file_counts.to_frame('count'))

# Year range within each file
print('\nYear range per source file:')
display(
    df_raw.groupby('source_file')['year_fix']
    .agg(['min', 'max', 'nunique'])
    .rename(columns={'min': 'year_min', 'max': 'year_max', 'nunique': 'n_years'})
)

In [ ]:
# ── Basic descriptive statistics ───────────────────────────────────────────────
numeric_cols = df_raw.select_dtypes(include='number').columns.tolist()
display(df_raw[numeric_cols].describe().round(3))

---
## 3. Target Variable Analysis — `weight_grams`

Clinical thresholds:
- **VLBW** (Very Low Birth Weight): < 1 500 g
- **LBW** (Low Birth Weight): < 2 500 g  
- **Normal**: 2 500 – 4 000 g
- **Macrosomia**: > 4 000 g

In [ ]:
# ── Convert target to grams (for clean df) ─────────────────────────────────────
target = df_feat['weight_grams'].dropna()

# Summary statistics
stats_dict = {
    'n': len(target),
    'mean (g)': target.mean(),
    'median (g)': target.median(),
    'std (g)': target.std(),
    'min (g)': target.min(),
    'max (g)': target.max(),
    'Q1 (g)': target.quantile(0.25),
    'Q3 (g)': target.quantile(0.75),
    'skewness': target.skew(),
    'kurtosis': target.kurtosis(),
}
for k, v in stats_dict.items():
    print(f'  {k:<18}: {v:>10.1f}')

In [ ]:
# ── Clinical category breakdown ────────────────────────────────────────────────
vlbw_pct  = (target < 1500).mean() * 100
lbw_pct   = (target < 2500).mean() * 100
normal_pct= ((target >= 2500) & (target <= 4000)).mean() * 100
macro_pct = (target > 4000).mean() * 100

print('Clinical weight categories:')
print(f'  VLBW  (<1500g):       {vlbw_pct:.2f}%')
print(f'  LBW   (<2500g):       {lbw_pct:.2f}%')
print(f'  Normal (2500–4000g):  {normal_pct:.2f}%')
print(f'  Macrosomia (>4000g):  {macro_pct:.2f}%')

In [ ]:
# ── Target distribution plots ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram + KDE
ax = axes[0]
ax.hist(target, bins=80, color='steelblue', alpha=0.7, density=True)
target.plot.kde(ax=ax, color='darkblue', linewidth=2)
for thresh, label, color in [
    (1500, 'VLBW', 'red'),
    (2500, 'LBW',  'orange'),
    (4000, 'Macro','green'),
]:
    ax.axvline(thresh, color=color, linestyle='--', linewidth=1.5, label=label)
ax.set_xlabel('Birth Weight (g)')
ax.set_ylabel('Density')
ax.set_title('Birth Weight Distribution')
ax.legend(fontsize=9)

# Box plot by sex
ax = axes[1]
male_w   = df_feat.loc[df_feat['is_male'] == 1, 'weight_grams'].dropna()
female_w = df_feat.loc[df_feat['is_male'] == 0, 'weight_grams'].dropna()
ax.boxplot([female_w, male_w], labels=['Female', 'Male'], notch=True, patch_artist=True,
           boxprops=dict(facecolor='lightblue'),
           medianprops=dict(color='darkblue', linewidth=2))
ax.set_ylabel('Birth Weight (g)')
ax.set_title('Birth Weight by Sex')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Q-Q plot for normality check
ax = axes[2]
sample = target.sample(min(5000, len(target)), random_state=42)
stats.probplot(sample, dist='norm', plot=ax)
ax.set_title('Q-Q Plot (Normality Check)')
ax.get_lines()[0].set(markersize=2, alpha=0.4, color='steelblue')
ax.get_lines()[1].set(color='red', linewidth=1.5)

plt.tight_layout()
plt.suptitle('Target Variable: Birth Weight (grams)', y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## 4. Feature Distributions

In [ ]:
# ── Numerical feature distributions ───────────────────────────────────────────
num_features = [
    'gestation_weeks', 'mother_age', 'weight_gain_pounds',
    'father_age', 'plurality', 'ever_born',
    'born_alive_alive', 'born_alive_dead', 'born_dead',
]
num_features = [c for c in num_features if c in df_feat.columns]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(num_features):
    ax = axes[i]
    data = df_feat[col].dropna()
    ax.hist(data, bins=40, color='steelblue', alpha=0.7)
    ax.set_title(f'{col}\n(n={len(data):,}, missing={df_feat[col].isna().mean()*100:.1f}%)')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical / binary feature distributions ─────────────────────────────────
cat_features = ['is_male', 'mother_married', 'plurality', 'record_weight']
cat_features = [c for c in cat_features if c in df_feat.columns]

fig, axes = plt.subplots(1, len(cat_features), figsize=(14, 4))
if len(cat_features) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_features):
    vc = df_feat[col].value_counts(dropna=False).sort_index()
    ax.bar(vc.index.astype(str), vc.values, color='coral', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    for patch, val in zip(ax.patches, vc.values):
        ax.text(patch.get_x() + patch.get_width()/2, patch.get_height() + 0.5,
                f'{val:,}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Categorical Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Box plots: key features grouped by LBW category ──────────────────────────
df_feat['lbw_category'] = pd.cut(
    df_feat['weight_grams'],
    bins=[0, 1500, 2500, 4000, 9999],
    labels=['VLBW\n(<1500g)', 'LBW\n(1500–2500g)', 'Normal\n(2500–4000g)', 'Macrosomia\n(>4000g)']
)

box_features = ['gestation_weeks', 'mother_age', 'weight_gain_pounds']
box_features = [c for c in box_features if c in df_feat.columns]

fig, axes = plt.subplots(1, len(box_features), figsize=(14, 5))

for ax, col in zip(axes, box_features):
    groups = [df_feat.loc[df_feat['lbw_category'] == cat, col].dropna()
              for cat in df_feat['lbw_category'].cat.categories]
    ax.boxplot(groups, labels=df_feat['lbw_category'].cat.categories,
               patch_artist=True,
               boxprops=dict(facecolor='lightcoral', alpha=0.7),
               medianprops=dict(color='darkred', linewidth=2))
    ax.set_title(f'{col} by Weight Category')
    ax.set_ylabel(col)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Feature Distributions by Birth Weight Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Missing Value Analysis

In [ ]:
# ── Missing value percentages ──────────────────────────────────────────────────
missing = df_raw.isnull().mean().mul(100).sort_values(ascending=False)
missing_df = missing[missing > 0].to_frame('missing_%').round(2)
print(f'Columns with missing values: {len(missing_df)}')
display(missing_df)

In [ ]:
# ── Missing value bar chart ────────────────────────────────────────────────────
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#d62728' if v > 50 else '#ff7f0e' if v > 10 else '#1f77b4'
              for v in missing_df['missing_%']]
    ax.barh(missing_df.index, missing_df['missing_%'], color=colors, alpha=0.85)
    ax.axvline(10, color='orange', linestyle='--', alpha=0.7, label='10% threshold')
    ax.axvline(50, color='red',    linestyle='--', alpha=0.7, label='50% threshold')
    ax.set_xlabel('Missing Values (%)')
    ax.set_title('Missing Values per Column (Raw Data)', fontweight='bold')
    ax.legend()
    for i, (col, row) in enumerate(missing_df.iterrows()):
        ax.text(row['missing_%'] + 0.5, i, f"{row['missing_%']:.1f}%",
                va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Missingno matrix (pattern visualization) ───────────────────────────────────
# Only include columns that have some missingness
cols_with_missing = missing[missing > 0].index.tolist()
if cols_with_missing:
    sample_for_msno = df_raw[cols_with_missing].sample(
        min(5000, len(df_raw)), random_state=42
    )
    fig, ax = plt.subplots(figsize=(12, 5))
    msno.matrix(sample_for_msno, ax=ax, sparkline=False, fontsize=9, color=(0.27, 0.52, 0.71))
    ax.set_title('Missing Value Pattern Matrix (sample n=5,000)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No columns with missing values found in raw data.')

In [ ]:
# ── Missing values per source file ────────────────────────────────────────────
key_cols = ['weight_pounds', 'gestation_weeks', 'weight_gain_pounds',
            'father_age', 'alcohol_use', 'cigarette_use']
key_cols = [c for c in key_cols if c in df_raw.columns]

missing_by_file = (
    df_raw.groupby('source_file')[key_cols]
    .apply(lambda g: g.isnull().mean() * 100)
    .round(1)
)
print('Missing % by source file (key columns):')
display(missing_by_file)

---
## 6. Correlation Analysis

In [ ]:
# ── Pearson correlation matrix (numerical features vs target) ─────────────────
corr_features = [
    'weight_grams', 'gestation_weeks', 'mother_age', 'weight_gain_pounds',
    'father_age', 'plurality', 'born_alive_alive', 'ever_born',
    'is_male', 'mother_married', 'parity',
]
corr_features = [c for c in corr_features if c in df_feat.columns]

corr_matrix = df_feat[corr_features].corr(method='pearson')

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0, ax=ax,
    annot_kws={'size': 8}, linewidths=0.5, square=True
)
ax.set_title('Pearson Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Ranked correlations with target ───────────────────────────────────────────
target_corr = corr_matrix['weight_grams'].drop('weight_grams').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2196F3' if v > 0 else '#F44336' for v in target_corr]
ax.barh(target_corr.index, target_corr.values, color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Birth Weight')
ax.set_title('Feature Correlations with Target (weight_grams)', fontweight='bold')
for i, (feat, val) in enumerate(target_corr.items()):
    ax.text(val + (0.005 if val >= 0 else -0.005), i,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 5 positive predictors:')
print(target_corr[target_corr > 0].head(5).to_string())
print('\nTop 5 negative predictors:')
print(target_corr[target_corr < 0].head(5).to_string())

In [ ]:
# ── Scatter plots: top 4 features vs target ───────────────────────────────────
top_features = target_corr.abs().head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

sample_scatter = df_feat.sample(min(10000, len(df_feat)), random_state=42)

for ax, feat in zip(axes, top_features):
    x = sample_scatter[feat].dropna()
    y = sample_scatter.loc[x.index, 'weight_grams'].dropna()
    common = x.index.intersection(y.index)
    ax.scatter(x[common], y[common], alpha=0.15, s=10, color='steelblue')
    # Trend line
    try:
        m, b = np.polyfit(x[common].astype(float), y[common].astype(float), 1)
        xline = np.linspace(x[common].min(), x[common].max(), 100)
        ax.plot(xline, m * xline + b, color='red', linewidth=2)
    except Exception:
        pass
    r = target_corr.get(feat, 0)
    ax.set_title(f'{feat}  (r = {r:.3f})')
    ax.set_xlabel(feat)
    ax.set_ylabel('weight_grams')

plt.suptitle('Top Features vs. Birth Weight (scatter + trend)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Temporal Trend Analysis

In [ ]:
# ── Mean birth weight by year ──────────────────────────────────────────────────
yearly = (
    df_feat.groupby('year_fix')['weight_grams']
    .agg(['mean', 'median', 'std', 'count'])
    .rename(columns={'mean': 'mean_g', 'median': 'median_g', 'std': 'std_g', 'count': 'n'})
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Mean birth weight trend
ax = axes[0, 0]
ax.plot(yearly.index, yearly['mean_g'], marker='o', markersize=4, color='steelblue', linewidth=2)
ax.fill_between(yearly.index, yearly['mean_g'] - yearly['std_g'],
                yearly['mean_g'] + yearly['std_g'], alpha=0.15, color='steelblue')
ax.set_title('Mean Birth Weight Over Years')
ax.set_ylabel('Birth Weight (g)')
ax.set_xlabel('Year')

# LBW prevalence trend
ax = axes[0, 1]
lbw_yearly = df_feat.groupby('year_fix').apply(
    lambda g: (g['weight_grams'] < 2500).mean() * 100
)
ax.plot(lbw_yearly.index, lbw_yearly.values, marker='o', markersize=4, color='coral', linewidth=2)
ax.set_title('LBW Prevalence Over Years (< 2500g)')
ax.set_ylabel('LBW Rate (%)')
ax.set_xlabel('Year')

# Preterm birth trend
ax = axes[1, 0]
if 'gestation_preterm' in df_feat.columns:
    preterm_yearly = df_feat.groupby('year_fix')['gestation_preterm'].mean().mul(100)
    ax.plot(preterm_yearly.index, preterm_yearly.values, marker='o', markersize=4,
            color='mediumpurple', linewidth=2)
    ax.set_title('Preterm Birth Rate Over Years (< 37 weeks)')
    ax.set_ylabel('Preterm Rate (%)')
    ax.set_xlabel('Year')

# Records per year
ax = axes[1, 1]
ax.bar(yearly.index, yearly['n'], color='lightsteelblue', alpha=0.85)
ax.set_title('Records per Year (sample)')
ax.set_ylabel('Record Count')
ax.set_xlabel('Year')

plt.suptitle('Temporal Trends in Birth Weight (Sample Data)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Distributional shift: KDE by era ──────────────────────────────────────────
# Check whether the weight distribution shifts meaningfully across eras
fig, ax = plt.subplots(figsize=(10, 5))

era_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
eras = sorted(df_feat['source_year'].unique())

for era, color in zip(eras, era_palette):
    subset = df_feat.loc[df_feat['source_year'] == era, 'weight_grams'].dropna()
    if len(subset) > 100:
        subset.plot.kde(ax=ax, label=f'Era {era} (n={len(subset):,})',
                        color=color, linewidth=2)

ax.axvline(2500, color='orange', linestyle='--', linewidth=1.5, label='LBW threshold')
ax.axvline(4000, color='green',  linestyle='--', linewidth=1.5, label='Macrosomia')
ax.set_xlabel('Birth Weight (g)')
ax.set_ylabel('Density')
ax.set_title('Birth Weight Distribution Shift Across Eras', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Data Leakage Check

In [ ]:
# ── Verify leakage columns are NOT in clean data ───────────────────────────────
leakage_cols = ['apgar_1min', 'apgar_5min']

print('=== Data Leakage Audit ===')
print(f'Leakage columns in RAW data:   {[c for c in leakage_cols if c in df_raw.columns]}')
print(f'Leakage columns in CLEAN data: {[c for c in leakage_cols if c in df_feat.columns]}')

if not any(c in df_feat.columns for c in leakage_cols):
    print('\n✅ PASS: No leakage columns present in processed feature set.')
else:
    print('\n❌ FAIL: Leakage columns detected in processed data!')

# Show Apgar correlation in raw data (to quantify the leakage risk)
raw_with_weight = df_raw.copy()
raw_with_weight['weight_grams'] = raw_with_weight['weight_pounds'] * 453.592
for col in leakage_cols:
    if col in df_raw.columns:
        r = raw_with_weight[['weight_grams', col]].dropna().corr().iloc[0, 1]
        print(f'  Correlation of {col} with weight_grams (in raw): r = {r:.3f}')

---
## 9. Key Findings Summary

> This section summarises the most important EDA findings for the `docs/data_report.md` deliverable.

In [ ]:
# ── Auto-generate findings summary ────────────────────────────────────────────
print('=' * 60)
print('EDA KEY FINDINGS SUMMARY')
print('=' * 60)

target_clean = df_feat['weight_grams'].dropna()
print(f'\n📊 Target Variable (weight_grams):')
print(f'   Mean:     {target_clean.mean():,.0f} g')
print(f'   Median:   {target_clean.median():,.0f} g')
print(f'   Std:      {target_clean.std():,.0f} g')
print(f'   Skewness: {target_clean.skew():.3f} (slightly left-skewed)')
print(f'   LBW prevalence:   {(target_clean < 2500).mean()*100:.2f}%')
print(f'   VLBW prevalence:  {(target_clean < 1500).mean()*100:.2f}%')
print(f'   Macrosomia:       {(target_clean > 4000).mean()*100:.2f}%')

print(f'\n❓ Missing Data (top 5 most missing columns in raw data):')
top_missing = df_raw.isnull().mean().mul(100).sort_values(ascending=False).head(5)
for col, pct in top_missing.items():
    print(f'   {col:<30}: {pct:.1f}% missing')

print(f'\n📈 Top Correlates with Birth Weight:')
if 'weight_grams' in corr_matrix.columns:
    tc = corr_matrix['weight_grams'].drop('weight_grams').sort_values(key=abs, ascending=False)
    for feat, val in tc.head(6).items():
        direction = '▲' if val > 0 else '▼'
        print(f'   {direction} {feat:<25}: r = {val:.3f}')

print(f'\n✅ Data Leakage: apgar_1min and apgar_5min excluded from feature set.')
print(f'\n📅 Temporal Trends: see charts in Section 7 above.')
print(f'\n📏 Sample used for EDA: {NROWS_PER_FILE:,} rows/file × 5 files = ~{len(df_raw):,} rows')
print('=' * 60)